In [ ]:
#imported from Pranav Sanghavi's notebook -- 
import glob 
import os
import time
import h5py
import matplotlib.pylab as plt
import numpy as np
import numpy.ma as ma
from matplotlib.offsetbox import AnchoredText
from matplotlib.pylab import colorbar 
from skrf import Network, Frequency
from platform import python_version
from rawice import *
import math
import psutil
import matplotlib.cm as cm
#importing for Will's function --  
from scipy import interpolate
import datetime
from scipy.signal import square
from scipy.stats import pearsonr
import pandas
import time_utils as tu
from mpl_toolkits.axes_grid1 import make_axes_locatable

style = 'fast'
tick_dir = 'in'
plt.style.use(style)
plt.rcParams['xtick.direction'] = tick_dir
plt.rcParams['ytick.direction'] = tick_dir
plt.rcParams['axes.linewidth'] = 1
plt.rcParams['font.size'] = 15
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.5
plt.rcParams.update({
    "text.usetex": False,
    "font.family": "serif"})

###################################################################################
# Analog chain labels
###################################################################################

#pathtoacfile = "/home/observer/tone_jupyter/anachainz.csv"
#ac = np.genfromtxt(pathtoacfile, delimiter=",", names=True,
#                   dtype=(int, int, 'U5', 'U5', int, 'U2', int))
#inputs = ac['feed']

def get_beam_fwhm(d=6.0, nu=600):
    fwhm = constants.c/(nu*1e6)/d   # lambda/D
    return np.rad2deg(fwhm * 1.22)

def gaussian(x, a, mean, sigma):
    val = a * np.exp(-(x - mean)**2 / sigma**2)
    return val

def scale_zero_one(x):
    return (x - x.min())/(x.max() - x.min())

def seconds_to_degrees(seconds, declination=22.0145):
    """
        This function converts seconds to degrees of sky motion at a latitude 
        default TONE latitude -- default for crab.
    """
    return seconds * np.cos(np.deg2rad(declination)) / 240

def read_corr_mode_h5(file_path, f_start=None, f_stop=None):
    """
        This function reads all hdf5 files.
    """
    class correlator_data:
        def __init__(self, vis, time, sat, index_map):
            self.vis = vis
            self.time = time
            self.sat = sat
            self.freq = index_map["freq"][:]
            self.prod = index_map['prod'][:]
    files = glob.glob(file_path + "*[!.lock]")
    files.sort()
    
    #grab different sets of files.  
    if f_start is not None:
        if f_stop is not None:
            files = files[f_start:f_stop]
        else:
            files = files[f_start:]
    if f_start is None:
        if f_stop is not None:
            files = files[:f_stop]

    try:
        t = 0
        for x in files:
            print(f"Reading File: {x}")
            f = h5py.File(x, "r")
            index_map = f['index_map']
            if(t == 0):
                vis = f['vis'][:]
                time = index_map['time'][:]
                sat = f['sat'][:]
                print("Appended vis data and time stream from %s" % (x))
                t = t + 1
            else:
                time = np.append(time, index_map['time'][:])
                vis = np.append(vis, f['vis'][:], axis=0)
                sat = np.append(sat, f['sat'][:], axis=0)
                print("Appended vis data and time stream from %s" % (x))
        
        return correlator_data(
            vis,
            time,
            sat,
            index_map
        )
    finally:
        print("Done!!!!!!!")

def get_mag_phase(corr_dat):
    """
        This function returns magnitude and phase in degrees of correlations.
    """
    return np.absolute(corr_dat), np.rad2deg(np.angle(corr_dat))

def plot_waterfall_corr_data(data, corr_indices):
    """
        This function plots waterfall across 1024 frequency channels.
        Frequency range: 400-800MHz
    """
    fig=plt.figure(figsize=(20,15))
    dim1 = math.floor(math.sqrt(len(corr_indices)))
    dim2 = math.ceil(len(corr_indices)/dim1)
    for i,auto in enumerate(corr_indices):
        ax=plt.subplot(dim1, dim2, i + 1)
        wfall = data[:, :, auto] # auto -> corr index 
        im=plt.imshow(wfall, vmin = np.nanpercentile(wfall,5), vmax =np.nanpercentile(wfall,95), cmap='magma', aspect='auto')
        plt.xlabel("freq index")
        plt.ylabel("time index")
        plt.title(f"corr {corr_data.prod[auto]}")
        divider=make_axes_locatable(ax)
        cax=divider.append_axes("right", size="5%", pad=0.05)
        cbar=fig.colorbar(im,cax=cax)
        cbar.set_label('Power [$ADU^2$]')
    plt.tight_layout()
    #plt.savefig(f"/Users/kalyanibhopi94/Documents/kalyanib_dns/corr_dns_plots/drone_testing_4x2_Apr2025/Apr07...246Z/switched_mode_reset/waterfall_{f_start}_{f_stop}.pdf",
    #            dpi=300)

    
def plot_freq_channel_corr_data(pow_mag, freq_index, freq, corr_indices):
    """
        This function plots phase for a given frequency index for given time window.
    """
    plt.figure(figsize=(20, 20))
    dim1 = math.floor(math.sqrt(len(corr_indices)))
    dim2 = math.ceil(len(corr_indices)/dim1)
    for index, corr_index in enumerate(corr_indices):
            plt.subplot(dim1, dim2, index + 1)
            plt.title(f"Corr: {corr_data.prod[corr_index]}")
            plt.plot(pow_mag[:, freq_index, corr_index])
    plt.suptitle(f"Frequency {freq} MHz")
    plt.tight_layout()
    
def plot_phase_corr_data(pow_phase, corr_indices, t_ind_arr):
    """
        This function plots phase for a given time index across frequency range of 400-800MHz.
    """
    colorsarr=cm.magma(np.linspace(0,1,2048))
    for i, t_index in enumerate(t_ind_arr):
        fig= plt.figure(figsize=(25, 10))
        dim1 = math.floor(math.sqrt(len(corr_indices)*len(t_ind_arr)))
        dim2 = math.ceil(len(corr_indices)*len(t_ind_arr)/dim1)
        for index, corr_index in enumerate(corr_indices):
                ax = plt.subplot(dim2, dim1, i + 1)
                ax.plot(pow_phase[t_index,:, corr_index], c=colorsarr[300], label=f'time index {t_index}')
                ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
                plt.xlabel(f"frequency index")
                plt.ylabel(f"phase (degrees)")
                plt.title(f"Corr: {corr_data.prod[corr_index]}")
                #plt.suptitle(f"Correlated noise phase")
        plt.tight_layout()
    #plt.savefig(f"/Users/kalyanibhopi94/Documents/kalyanib_dns/corr_dns_plots/drone_testing_4x2_Apr2025/Apr07...246Z/switched_mode_reset/phase_{f_start}_{f_stop}.pdf",
    #            dpi=300)


def plot_freq_channel_corr_data(pow_phase, freq_index, corr_indices):
    """
        This function plots phase variation for a single frequency bin as a function of time.  

    """
    plt.figure(figsize=(25, 10))
    dim1 = math.floor(math.sqrt(len(corr_indices)))
    dim2 = math.ceil(len(corr_indices)/dim1)
    frequency = corr_data.freq[freq_index]
    for index, corr_index in enumerate(corr_indices):
            plt.subplot(dim1, dim2, index + 1)
            plt.title(f"Corr: {corr_data.prod[corr_index]} : {frequency[0][0]} MHz")
            plt.plot(pow_phase[:,freq_index, corr_index])
            plt.xlabel("time indices")
            plt.ylabel("Phase (deg)")
    plt.savefig(f"/Users/kalyanibhopi94/Downloads/June2025_pulsing_mode_time_stability_chamber/investigate_plots/21.png", dpi=300)       
    #plt.suptitle(f"Frequency {freq} MHz")
    plt.tight_layout()

In [ ]:
print("current python version is:",python_version())
print("current numpy version is:",np.__version__)

## Reading digital gains for all the correlator channels 

In [ ]:
f = h5py.File("/Users/kalyanibhopi94/Documents/research/Projects/digital_cal_source/data_raw/iceboard_data/digital_gains/20250319T171034Z_WVUc52_digitalgain/00000057.h5","r")

#Plotting digital gains -- 
print(list(f.attrs))
print(f.attrs['acquisition_name'])
print(f.attrs['archive_version'])
print(f.attrs['collection_server'])
print(f.attrs['git_version_tag'])
print(f.attrs['instrument_name'])
print(f.attrs['notes'])
print(f.attrs['system_user'])
print(f.attrs['type'])
print(f.attrs['version'])

print(list(f.items()))
print((f['compute_time']))
print((f['index_map']))
print((f['gain_coeff']))
print((f['gain_exp']))

inputs = (f['index_map']['input'][:])
frqs = f['index_map']['freq'][:]
gaincoeff = f['gain_coeff'][()]
gainexp = f['gain_exp'][()]
gain = gaincoeff.real*np.exp2(gainexp)

plt.figure(figsize=[16,18])
print(f"Calculating digital gains of the channels...")
i = 0
for i in np.arange(0,4,1):  
    plt.subplot(8,2,i+1)
    #Digital gain of every channel
    plt.scatter(frqs['centre'], gain[0][:,i]) # i referes to input channel (total 16 ch on iceboard); 
    #plt.title( " %s and gain_exp = %s and rms %s for rms i/p %s" %(inputs[i], gainexp[:,i], rms[i][2], rms[i][0][2]))
    plt.title( " %s " %(inputs[i]))
plt.tight_layout()

In [ ]:
# gains related stats -- 

print(gainexp)
print(gaincoeff[0,:,1])
gain = gaincoeff.real*np.exp2(gainexp)
print(gain.shape)
print(gain[0,:,1])
print(gain[0,:,1].shape)
print(frqs['centre'])
print(inputs)

## Analysing raw adc rms of the whole data run; Plotting Histogram of the signal received per correlator channel; From Pranav's "analyse_raw_adc-checkpoint.ipynb" notebook

In [ ]:
folder_path = "/Users/kalyanibhopi94/Downloads/iceboard_data/corr_data/20250609T180246Z_WVUc52/raw_acq/"
try:
    file_name = get_newest_file(folder_path)
    raw_acq(file_name, False)
except OSError:
    file_name = get_second_newest_file(folder_path)
    raw_acq(file_name, False)

In [ ]:
crate_number = 0 
slot_number = 0

figrms = plt.figure(figsize=(30, 30))
figfft = plt.figure(figsize=(30, 30))
fighist = plt.figure(figsize=(30, 30))

figrms.suptitle(f"ADC RMS START: {raw_acq.start_time} STOP: {raw_acq.end_time}\n crate no.:{crate_number} slot no.:{crate_number}", y=0.92,
ha='center')
figfft.suptitle(f"FFT MAG START: {raw_acq.start_time} STOP:{raw_acq.end_time} \n crate no.:{crate_number} slot no.:{crate_number}", y=0.92,
ha='center')
fighist.suptitle(f"ADC Histogram START: {raw_acq.start_time} STOP:{raw_acq.end_time} \n crate no.:{crate_number} slot no.:{crate_number} ", y=0.92,
ha='center')


rms_above_threshold = {}
rms_below_threshold = {}

for i in range(4):
    input_to_check = [crate_number, slot_number, i]
    raw = raw_acq.check_input(input_to_check)
    rms = raw.rms
    if np.median(rms) > 60:
        rms_above_threshold[i] = "ADC RMS above 60"
    if np.median(rms) < 8:
        rms_below_threshold[i] = "ADC RMS below 8"
    fft = raw.mag_fft
    ts = raw.time_fpga_count
    times = (ts - ts[0])*2.56e-6/60
    axrms = figrms.add_subplot(4, 4, i+1)
    axrms.scatter(times, rms, s=1, c='k', rasterized=True)
    axrms.set_title(f'input: {i}')
    axrms.set_xlabel('minutes')
    axrms.set_ylabel('adc units')
    axrms.axhline([2], lw=0.5, c='r')
    axrms.grid(axis='both', linestyle='-', linewidth=1)
    
    axfft = figfft.add_subplot(4, 4, i+1)
    axfft.imshow(
        fft,
        aspect='auto',
        vmin=np.percentile(fft, 5),
        vmax=np.percentile(fft, 95),
        extent=[800, 400, times[-1], times[0]],
        interpolation='none',
        rasterized=True
    )
    axfft.set_title(f'input: {i}')
    axfft.set_ylabel("minutes")
    axfft.set_xlabel("MHz")
    
    inp0 = np.where(raw_acq.adc_input[:] == i)[0]
    ant0_data = raw_acq.timestream[:][inp0]
    ant0_data = ant0_data[:]
    axhist = fighist.add_subplot(4, 4, i+1)
    hist, bin_edges = np.histogram(ant0_data, bins=256,  density=True)
    axhist.step(bin_edges[1:], hist, where='mid', c='k', rasterized=True)
    axhist.set_title(f'input: {i}')
    axhist.set_xlim([-129, 129])
    axhist.set_ylim(0,0.1)
    
#pp = PdfPages(f"{SAVE_DIR}{raw_acq.start_time}_{raw_acq.end_time}_raw_acq.pdf")
#pp.savefig(figrms)
#pp.savefig(figfft)
#pp.savefig(fighist)
#pp.close()

## Analysis of correlator data for correlation powers, waterfall and phase

In [ ]:
#Reading correlation files -- 
file_path = '/Users/kalyanibhopi94/Downloads/drone_flight8_NWgrid_NWpol_42db_jul2_1930/*' 
f_start=1
f_stop=4
corr_data = read_corr_mode_h5(file_path, f_start, f_stop) 
print("getting auto and cross-correlation magnitudes and phase")
pow_mag, pow_phase = get_mag_phase(corr_data.vis)
print("done getting pow_mag and pow_phase")
sat_data = corr_data.sat.real

corr_size=corr_data.vis[:,:,:].shape
print('corr data shape: {}'.format(corr_size)) # time bins - freq bins - corr indices

In [ ]:
# visibility data 
corr_data.vis[1][:][31] # correlation data having real and imag component 
print(corr_data.vis[:,:,:].shape) # time bins - freq bins - corr indices

In [ ]:
## Plotting real and imag correlated system noise powers against time index for a given set of frequencies -- 

tmin,tmax=[0,corr_size[0]]
freq_arr=[250,546,625,825]
corr_indices=[1]
dim1 = math.floor(math.sqrt(len(corr_indices)*len(freq_arr)))
dim2 = math.ceil(len(corr_indices)*len(freq_arr)/dim1)
fig = plt.figure(figsize=[50,40])
for index1,f_ind in enumerate(freq_arr):
    colorsarr=cm.magma(np.linspace(0,1,2048))
    freq=400+((1024-f_ind)*0.390625)
    for index2, corr_index in enumerate(corr_indices): 
        panel_ind = index1 * len(corr_indices) + index2 + 1 
        ax=plt.subplot(dim2, dim1, panel_ind)
        ax.plot(corr_data.vis.real[tmin:tmax,f_ind,corr_index],'o-', markersize=13, c=colorsarr[600], label='real')
        ax.plot(corr_data.vis.imag[tmin:tmax,f_ind,corr_index],'o-', markersize=13, c=colorsarr[1200], label='imag')
        ax.plot(pow_mag[tmin:tmax,f_ind,corr_index],'o-', markersize=13, c=colorsarr[800], label='abs')
        #ax.set_xlim(100,300)
        #ax.set_ylim(35000,42000)
        ax.legend(fontsize=40)
        plt.xticks(fontsize=40)
        plt.yticks(fontsize=40)
        plt.title(f"corr {corr_data.prod[corr_index]}", fontsize=40)
        plt.xlabel("time index", fontsize=40)
        plt.ylabel("corr power (arb)", fontsize=40)
    #plt.suptitle(f"Absolute power for frequency {freq}MHz", ha='center', fontsize=20)
    fig.tight_layout(rect=[0, 0.03, 1, 0.95])
#fig.savefig(f"/Users/kalyanibhopi94/Downloads/June2025_pulsing_mode_time_stability_chamber/investigate_plots/20.png", dpi=300)
    

delta_time = np.diff(corr_data.time["irigb_time"])[1]*1e-9 
print(f"Each time bin is: {delta_time} seconds")
print(f"total time bins (considered for the plots) are {len(corr_data.vis[tmin:tmax,0,0])}") 
print(f"lowest time bin {tmin} and highest time bin {tmax}")
print(f"{delta_time*len(corr_data.vis[tmin:tmax,0,0])} seconds of data (considered for the plots)") 

In [ ]:
# plotting waterfall --

plot_waterfall_corr_data(pow_mag[:,:,:],[0,1,4])

In [ ]:
# Phase variation for a single frequency bin as a function of time -- calling a function

#frequency_index = np.arange(600,850,1)
#frequency_index = [250,546,625,825]
frequency_index = [825]
frequency = corr_data.freq[frequency_index]
print(f"Frequency: {frequency}")
plot_freq_channel_corr_data(pow_phase,frequency_index,[1])

In [ ]:
#Phase for a set of time indices as a function of frequency -- calling a function

t_ind_arr= [182] 
corr_indices=[1] 

plot_phase_corr_data(pow_phase[:,:,:], corr_indices, t_ind_arr)